# Desafio 5 — Ciência de Dados 1

**Aluno(a): Pedro Vyctor Carvalho de Almeida**

Conteúdo: **seleção e comparação de modelos, limiar de decisão e calibração** e **target
leakage e reprodutibilidade**.

Rode cada célula e **escreva sua resposta** no campo indicado. Você **não** precisa escrever
código — o objetivo é interpretar e comparar os resultados. Os arquivos `.csv` estão nesta
pasta.

> **Regra deste desafio:** toda afirmação que menciona um valor deve **trazer o valor**,
> copiado da saída da sua célula. Os números mudam de caderno para caderno.

São cinco exercícios de 2,0 pontos. Algumas células treinam florestas em 10 dobras e podem
levar alguns segundos: espere o `[*]` virar número antes de passar adiante.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, joblib
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     cross_validate, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_auc_score

def logistica():
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=2000))])

def avalia(y, p, corte, c_fn):
    """FP, FN, precisão, recall e custo (um FN vale c_fn FP) num corte."""
    tn, fp, fn, tp = confusion_matrix(y, (p >= corte).astype(int), labels=[0, 1]).ravel()
    precisao = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn)
    return dict(FP=int(fp), FN=int(fn), precisao=precisao, recall=recall,
                custo=int(fp + c_fn * fn))

cv = StratifiedKFold(10, shuffle=True, random_state=0)
print("ambiente pronto")

## Exercício 1 — Quatro candidatos, uma régua  *(2,0 pontos)*

A base `q_candidatos.csv` traz clientes de uma operadora; o alvo é `alvo = 1` para quem
**cancelou** o plano. Quatro candidatos passam pelas **mesmas 10 dobras**: o chute (sempre a
classe mais comum), uma árvore de profundidade 3, uma floresta e a regressão logística. A saída
traz o ROC-AUC (média e desvio entre as dobras) e a acurácia de cada um.

**(a)** Qual candidato tem o maior ROC-AUC? Dê a média e o desvio dele, e as médias dos outros
três.

**(b)** Use a regra da faixa de empate (a melhor média menos um desvio dela). Qual é o piso, quais
candidatos ficam dentro e quais ficam fora? O que isso diz sobre a diferença entre os de dentro?

**(c)** Qual modelo você entrega, e por quê? Se houver empate, diga qual é o critério de
desempate — e por que ele não é a terceira casa decimal.

**(d)** Olhe a acurácia do chute. Dê o valor e explique por que a comparação foi feita pelo
ROC-AUC e não pela acurácia.

In [ ]:
df = pd.read_csv("q_candidatos.csv")
X = df.drop(columns="alvo")
y = df["alvo"]
print("linhas:", len(y), "| colunas:", X.shape[1], "| cancelaram:", f"{y.mean():.1%}")

candidatos = {
    "chute":     DummyClassifier(strategy="most_frequent"),
    "árvore":    DecisionTreeClassifier(max_depth=3, random_state=0),
    "floresta":  RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=0),
    "logística": logistica(),
}

res = {}
print("\ncandidato    ROC-AUC   desvio   acurácia")
for nome, modelo in candidatos.items():
    r = cross_validate(modelo, X, y, cv=cv, scoring=["roc_auc", "accuracy"])
    res[nome] = (r["test_roc_auc"].mean(), r["test_roc_auc"].std(), r["test_accuracy"].mean())
    print(f"{nome:10s}   {res[nome][0]:.4f}   {res[nome][1]:.4f}   {res[nome][2]:.4f}")

melhor = max(res, key=lambda k: res[k][0])
piso = res[melhor][0] - res[melhor][1]
print(f"\nmelhor ROC-AUC: {melhor}  {res[melhor][0]:.4f} ± {res[melhor][1]:.4f}")
print(f"piso da faixa de empate: {piso:.4f}")
print("dentro da faixa:", [n for n in res if res[n][0] >= piso])
print("fora da faixa  :", [n for n in res if res[n][0] < piso])

**Resposta:**

_(escreva aqui)_

## Exercício 2 — O corte que vem de fábrica e o corte que custa menos  *(2,0 pontos)*

A base `q_triagem.csv` traz pacientes de um pronto-atendimento; `alvo = 1` marca **alto risco**.
A logística já foi escolhida. Um quarto da base foi guardado como teste no começo; as
probabilidades de **validação** vêm da validação cruzada no restante. Para a área de saúde,
**um falso negativo vale 6 falsos positivos** — e, em outra conversa, a área pediu
**recall de pelo menos 0,90**.

**(a)** Na tabela dos quatro cortes, o que acontece com a precisão e com o recall quando o corte
sobe? Dê os valores nos cortes 0,15 e 0,70. Existe um corte que melhore os dois?

**(b)** Na varredura do custo, onde fica o mínimo, quanto ele custa e quanto custa o corte 0,50?
Dê a economia em porcentagem e diga por que o 0,50 não é uma escolha neutra.

**(c)** Para a meta de recall, qual é o **maior** limiar que a cumpre? Dê o limiar, o recall e a
precisão nesse ponto, e explique por que se escolhe o maior e não o menor.

**(d)** Na auditoria, compare o custo por paciente na validação e no teste, nos dois cortes. Por
que a linha do 0,50 é o controle do experimento, e por que o corte foi escolhido na validação e
não no teste?

In [ ]:
df = pd.read_csv("q_triagem.csv")
X = df.drop(columns="alvo")
y = df["alvo"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print("treino:", len(y_tr), "| teste:", len(y_te), "| alto risco no treino:", f"{y_tr.mean():.1%}")

C_FN = 6          # um falso negativo vale C_FN falsos positivos
META = 0.90       # recall mínimo pedido pela área

p_va = cross_val_predict(logistica(), X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]

print("\ncorte    FP   FN   precisão   recall   custo")
for c in [0.15, 0.30, 0.50, 0.70]:
    r = avalia(y_tr, p_va, c, C_FN)
    print(f"{c:.2f}   {r['FP']:4d} {r['FN']:4d}    {r['precisao']:.3f}     {r['recall']:.3f}   {r['custo']:5d}")

cortes = np.round(np.arange(0.01, 1.00, 0.01), 2)
custos = [avalia(y_tr, p_va, c, C_FN)["custo"] for c in cortes]
i = int(np.argmin(custos))
c_min, k_min = cortes[i], custos[i]
k_def = avalia(y_tr, p_va, 0.50, C_FN)["custo"]
print(f"\ncusto mínimo: corte {c_min:.2f}  custo {k_min}")
print(f"corte 0,50  : custo {k_def}")
print(f"economia    : {1 - k_min / k_def:.1%}")

precisao, recall, limiares = precision_recall_curve(y_tr, p_va)
cumprem = [(t, P, R) for t, P, R in zip(limiares, precisao[:-1], recall[:-1]) if R >= META]
t_meta, P_meta, R_meta = max(cumprem, key=lambda z: z[0])
print(f"\nmeta de recall 0.90: maior limiar {t_meta:.3f}  recall {R_meta:.3f}  precisão {P_meta:.3f}")

p_te = logistica().fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
print("\nauditoria — custo por paciente")
print("corte     validação   teste")
for c in [0.50, c_min]:
    kv = avalia(y_tr, p_va, c, C_FN)["custo"] / len(y_tr)
    kt = avalia(y_te, p_te, c, C_FN)["custo"] / len(y_te)
    print(f"{c:.2f}       {kv:.3f}     {kt:.3f}")

**Resposta:**

_(escreva aqui)_

## Exercício 3 — A coluna que entrega a resposta  *(2,0 pontos)*

Um banco quer prever, **no momento em que aprova o cartão**, quais clientes vão atrasar a
fatura nos 12 meses seguintes (`atrasou = 1`). A base `q_credito.csv` veio do data lake com
estas colunas:

| coluna | o que é | quando nasce |
|---|---|---|
| `renda_mensal` | renda declarada | no cadastro |
| `idade` | idade do cliente | no cadastro |
| `score_bureau` | pontuação do birô de crédito | consultada na aprovação |
| `comprometimento` | % da renda já comprometida com dívidas | no cadastro |
| `tempo_emprego` | anos no emprego atual | no cadastro |
| `dias_atraso_max` | maior atraso, em dias, nos 12 meses | ao longo dos 12 meses |
| `ligacoes_cobranca` | ligações da cobrança nos 12 meses | ao longo dos 12 meses |
| `limite_reduzido` | 1 se o banco reduziu o limite no ano | ao longo dos 12 meses |
| `nota_pesquisa` | nota da pesquisa de satisfação | no 6º mês |

**(a)** Faça a auditoria: quais colunas ficam e quais saem? Diga qual é a pergunta que decide.

**(b)** Dê a acurácia com todas as colunas, só com as que ficam e a do chute. Qual número vai
para o relatório do banco, e quanto ele ganha do chute?

**(c)** Na tabela "uma de cada vez", quais colunas inflam a acurácia, e qual praticamente não
mexe nela? Dê os valores. A que não mexe também sai? Por quê?

**(d)** Quais dos três sinais de alerta de vazamento aparecem nos seus números? Cite os valores.

In [ ]:
df = pd.read_csv("q_credito.csv")
y = df["atrasou"]
X = df.drop(columns="atrasou")
print("linhas:", len(y), "| atrasaram:", f"{y.mean():.1%}")

FICAM = ["renda_mensal", "idade", "score_bureau", "comprometimento", "tempo_emprego"]
DEPOIS = ["dias_atraso_max", "ligacoes_cobranca", "limite_reduzido", "nota_pesquisa"]

def acuracia_cv(colunas):
    return cross_val_score(logistica(), X[colunas], y, cv=cv, scoring="accuracy").mean()

acc_todas = acuracia_cv(FICAM + DEPOIS)
acc_ficam = acuracia_cv(FICAM)
acc_chute = cross_val_score(DummyClassifier(strategy="most_frequent"), X[FICAM], y,
                            cv=cv, scoring="accuracy").mean()
print(f"\ntodas as colunas : {acc_todas:.3f}")
print(f"só as que ficam  : {acc_ficam:.3f}")
print(f"chute            : {acc_chute:.3f}")

print("\numa de cada vez: as que ficam + a coluna")
print("coluna acrescentada    acurácia   diferença   correlação com o alvo")
uma_a_uma = {}
for col in DEPOIS:
    acc = acuracia_cv(FICAM + [col])
    corr = np.corrcoef(X[col], y)[0, 1]
    uma_a_uma[col] = (acc, corr)
    print(f"{col:20s}    {acc:.3f}     {acc - acc_ficam:+.3f}        {corr:+.3f}")

**Resposta:**

_(escreva aqui)_

## Exercício 4 — Ordena igual, promete diferente  *(2,0 pontos)*

A base `q_calibra.csv` traz pedidos de uma loja on-line; `alvo = 1` marca os pedidos que
**foram devolvidos**. A logística e uma floresta geram probabilidades de validação nas mesmas
10 dobras. A saída traz o ROC-AUC de cada uma, a tabela **previsto × observado** por faixa de
probabilidade e, como na questão do limiar, o corte de custo mínimo de cada modelo — aqui com
**um falso negativo valendo 6 falsos positivos**.

**(a)** Dê o ROC-AUC dos dois modelos. Pela **ordem** que dão aos pedidos, eles são diferentes?

**(b)** Na tabela previsto × observado, qual modelo é honesto e qual não é? Cite a faixa em que o
desonesto mais erra — o que ele prometeu e o que aconteceu — e diga se ele promete demais ou de
menos.

**(c)** Por que o ROC-AUC não enxerga o problema do item (b)? E o que você faz: calibra a floresta
ou troca de modelo? Justifique.

**(d)** Dê o corte de custo mínimo de cada modelo e o custo que a floresta teria se você
reaproveitasse nela o corte da logística. Por que o corte não pode ser levado de um modelo
para o outro?

In [ ]:
df = pd.read_csv("q_calibra.csv")
X = df.drop(columns="alvo")
y = df["alvo"]
print("linhas:", len(y), "| devolvidos:", f"{y.mean():.1%}")

C_FN = 6
floresta = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=0)
p_log = cross_val_predict(logistica(), X, y, cv=cv, method="predict_proba")[:, 1]
p_flo = cross_val_predict(floresta, X, y, cv=cv, method="predict_proba")[:, 1]
print(f"\nROC-AUC  logística {roc_auc_score(y, p_log):.4f}   floresta {roc_auc_score(y, p_flo):.4f}")
print(f"maior probabilidade dada: logística {p_log.max():.3f}   floresta {p_flo.max():.3f}")

faixas = [(0.0, 0.1), (0.1, 0.3), (0.3, 0.5), (0.5, 0.7), (0.7, 1.01)]
print("\n             ---- logística ----        ---- floresta -----")
print("faixa        n  prometeu  aconteceu     n  prometeu  aconteceu")
for a, b in faixas:
    linha = f"{a:.1f}-{min(b, 1.0):.1f}  "
    for p in (p_log, p_flo):
        m = (p >= a) & (p < b)
        if m.sum() > 0:
            linha += f"{m.sum():5d}   {p[m].mean():.3f}     {y[m].mean():.3f}    "
        else:
            linha += f"{0:5d}       —         —      "
    print(linha)

cortes = np.round(np.arange(0.01, 1.00, 0.01), 2)
k_log = [avalia(y, p_log, c, C_FN)["custo"] for c in cortes]
k_flo = [avalia(y, p_flo, c, C_FN)["custo"] for c in cortes]
c_log, c_flo = cortes[int(np.argmin(k_log))], cortes[int(np.argmin(k_flo))]
print(f"\ncorte de custo mínimo — logística {c_log:.2f} (custo {min(k_log)})   "
      f"floresta {c_flo:.2f} (custo {min(k_flo)})")
k_emprestado = avalia(y, p_flo, c_log, C_FN)["custo"]
print(f"floresta usando o corte {c_log:.2f} da logística: custo {k_emprestado} "
      f"({k_emprestado / min(k_flo) - 1:+.1%} em relação ao ótimo dela)")

**Resposta:**

_(escreva aqui)_

## Exercício 5 — Mesmo código, mesmo número — e o modelo inteiro no arquivo  *(2,0 pontos)*

A base `q_recompra.csv` traz clientes de um e-commerce; `alvo = 1` marca quem **comprou de
novo** em 90 dias. A primeira célula repete a mesma divisão treino/teste com 20 sementes
(`random_state` de 0 a 19) e mede a acurácia no teste de cada uma; depois mede a validação
cruzada. A segunda salva o Pipeline treinado com `joblib`, carrega de volta e pontua 20 clientes
novos de dois jeitos: com o Pipeline carregado e com **só o classificador**, refazendo a
padronização à mão nos próprios 20 clientes.

**(a)** Dê a menor e a maior acurácia (com as sementes), a diferença entre elas e a média.

**(b)** Um colega rodou as 20, viu a semente da maior acurácia e reportou esse número. O que está
errado nisso? O que você reportaria no lugar? Use os seus valores.

**(c)** O Pipeline carregado do arquivo dá as mesmas previsões do modelo original? O que o
`joblib.dump` gravou, além da regressão?

**(d)** No caminho "só o classificador", qual foi a maior diferença de probabilidade para um
mesmo cliente e quantos clientes trocaram de classe? Explique de onde vem a diferença e o que
deveria ter sido salvo.

In [ ]:
df = pd.read_csv("q_recompra.csv")
X = df.drop(columns="alvo")
y = df["alvo"]
print("linhas:", len(y), "| compraram de novo:", f"{y.mean():.1%}")

acc = []
for s in range(20):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=s)
    acc.append(logistica().fit(Xa, ya).score(Xb, yb))
    print(f"semente {s:2d}: acurácia no teste {acc[-1]:.3f}")
acc = np.array(acc)
print(f"\nmenor: {acc.min():.3f} (semente {acc.argmin()})   maior: {acc.max():.3f} (semente {acc.argmax()})")
print(f"diferença: {acc.max() - acc.min():.3f}   média: {acc.mean():.3f}")

vc = cross_val_score(logistica(), X, y, cv=cv, scoring="accuracy")
print(f"validação cruzada (10 dobras): {vc.mean():.3f} ± {vc.std():.3f}")

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
modelo = logistica().fit(X_tr, y_tr)
antes = modelo.predict_proba(X_te)[:, 1]

joblib.dump(modelo, "modelo_recompra.joblib")
carregado = joblib.load("modelo_recompra.joblib")
print("etapas do objeto salvo:", list(carregado.named_steps))
print("mesmas probabilidades depois de carregar?",
      np.array_equal(antes, carregado.predict_proba(X_te)[:, 1]))

novos = X_te.head(20)                                   # os 20 clientes novos
p_pipe = carregado.predict_proba(novos)[:, 1]

so_classificador = carregado.named_steps["clf"]
p_mao = so_classificador.predict_proba(StandardScaler().fit_transform(novos))[:, 1]

dif = np.abs(p_pipe - p_mao)
trocas = int(((p_pipe >= 0.5) != (p_mao >= 0.5)).sum())
k = int(dif.argmax())
print(f"\nmaior diferença de probabilidade: {dif.max():.3f}  (cliente {k}: "
      f"{p_pipe[k]:.3f} no Pipeline, {p_mao[k]:.3f} à mão)")
print(f"clientes que trocaram de classe: {trocas} de 20")

**Resposta:**

_(escreva aqui)_